# Graph Algorithms: Centrality, Paths & Communities
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/08_Graphs_Networks/graph_algorithms_centrality_community.ipynb)

Networks hide leaders, bridges and tribes. This notebook quantifies them on Zachary's Karate Club (the classic social network of 34 members that famously split in two) using NetworkX.

Covered: 5 centrality measures, shortest paths, connected components, community detection.

## 1. Load the famous karate club graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.karate_club_graph()
print(G.number_of_nodes(), "members,", G.number_of_edges(), "friendships")

pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(7, 5))
nx.draw(G, pos, node_size=350, node_color="lightblue", with_labels=True)
plt.title("Karate club social network"); plt.show()

## 2. Who matters? Five answers

In [ ]:
import pandas as pd

centralities = {
    "degree":      nx.degree_centrality(G),        # most connections
    "betweenness": nx.betweenness_centrality(G),   # bridges between groups
    "closeness":   nx.closeness_centrality(G),     # fastest to everyone
    "eigenvector": nx.eigenvector_centrality_numpy(G),  # connected to important ppl
    "pagerank":    nx.pagerank(G, alpha=0.85),     # Google's importance
}
scores = pd.DataFrame(centralities)
print(scores.sort_values("betweenness", ascending=False).head(5).round(3))
print("\nDifferent measures crown different leaders - pick by question:")

| Question | Measure |
|---|---|
| who has most friends? | degree |
| who connects rival groups? | betweenness |
| who spreads info fastest? | closeness |
| whose friends matter? | eigenvector / pagerank |

## 3. Shortest paths & structure

In [ ]:
print("diameter (max shortest path):", nx.diameter(G))
print("avg shortest path:", round(nx.average_shortest_path_length(G), 2))
print("path 0->33:", nx.shortest_path(G, 0, 33))
print("all short paths 0->33:", len(list(nx.all_shortest_paths(G, 0, 33))))

## 4. Community detection - predict the club split

In [ ]:
greedy = nx.community.greedy_modularity_communities(G)
louvain = nx.community.louvain_communities(G, seed=42)
print(f"greedy modularity found {len(greedy)} communities")
print(f"louvain found {len(louvain)} communities:", [sorted(c) for c in louvain])

truth = [G.nodes[n]["club"] for n in G]                # 'Mr. Hi' vs 'Officer'
from sklearn.metrics import adjusted_rand_score
pred_map = {n: i for i, c in enumerate(louvain) for n in c}
pred = [pred_map[n] for n in G]
print("agreement with real split (ARI):",
      round(adjusted_rand_score(truth, pred), 3))

In [ ]:
colors = [pred_map[n] for n in G]
plt.figure(figsize=(7, 5))
nx.draw(G, pos, node_color=colors, cmap="Set1", node_size=380, with_labels=True)
plt.title("Detected communities (matches the real club split!)"); plt.show()

## Takeaways
- Louvain recovered the historical split of the club purely from friendship edges.
- Betweenness nodes are your influencers/brokers - removing them fragments networks.
- Same toolkit applies to fraud rings, protein interactions, road networks, dependency graphs.
- For billion-edge graphs: Apache Spark GraphX/GraphFrames or cuGraph share these exact algorithms.